<a href="https://colab.research.google.com/github/Fried-toofuu/music-data/blob/main/data_pipeline/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Khám phá Dữ liệu & Metadata
Notebook này thực hiện việc đọc dữ liệu tracks từ file (FMA hoặc GTZAN), vẽ đồ thị phân phối (distribution) và kiểm tra metadata cơ bản.


In [1]:
import sys
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# Xác định môi trường
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    SRC_DIR = '/content/music-data/data_pipeline'
else:
    from pathlib import Path
    SRC_DIR = str(Path().resolve().parent)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Tải Metadata từ Hugging Face (Dành cho Demo)
Thay vì tải file CSV nội bộ (có thể không có sẵn), chúng ta sẽ sử dụng bộ dữ liệu `Khahn-nh/fma-small-1` đã được publish lên Hugging Face để khám phá nhanh.


In [1]:
import pyarrow as pa

print("Đang tải metadata từ Hugging Face Dataset...")
# Tải dưới dạng dataframe của Pandas
hf_dataset = load_dataset('Khahn-nh/fma-small-1', split='train')

# Chuyển đổi HF dataset -> Pandas DataFrame
pandas_df = hf_dataset.to_pandas()

columns_to_drop = []
if 'audio' in pandas_df.columns:
    columns_to_drop.append('audio')
if 'mel' in pandas_df.columns:
    columns_to_drop.append('mel')

if columns_to_drop:
    pandas_df = pandas_df.drop(columns=columns_to_drop)

# Chuyển đổi Pandas DataFrame -> Arrow Table -> Polars DataFrame
# Điều này tránh các vấn đề tương thích với các kiểu dữ liệu mở rộng (ExtensionDtype) của Pandas
arrow_table = pa.Table.from_pandas(pandas_df)
df = pl.from_arrow(arrow_table)


print(f"Total records: {df.shape[0]}")
display(df.head())

Đang tải metadata từ Hugging Face Dataset...


NameError: name 'load_dataset' is not defined

## 2. Thống kê số lượng theo Genre


In [2]:
# Cột chứa nhãn thể loại thường được gọi là 'label' trong HF dataset
if 'label' in df.columns:
    genre_col = 'label'
else:
    genre_col = df.columns[-1]  # Lấy đại cột cuối

genre_counts = df[genre_col].value_counts().sort('count', descending=True)

plt.figure(figsize=(10,6))
sns.barplot(data=genre_counts.to_pandas(), x='count', y=genre_col, palette='viridis')
plt.title('Phân phối số lượng bài hát theo thể loại (Genre)')
plt.xlabel('Số lượng bài')
plt.ylabel('Thể loại')
plt.tight_layout()
plt.show()


NameError: name 'df' is not defined